# Load a PDF and Inspect the Raw Output

In [1]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("chunking.pdf")
pages = loader.load()

print(f"Pages loaded: {len(pages)}")
print(f"\nFirst page content (first 500 chars):\n{pages[0].page_content[:500]}")
print(f"\nMetadata on page 0: {pages[0].metadata}")

/tmp/ipykernel_366068/1578966241.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader
/home/ml/miniconda3/envs/vllm/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Pages loaded: 5

First page content (first 500 chars):
RAG Chunking Demo
1
 Beispieldokument fuer RAG-Chunking
Ein kurzes, bewusst heterogenes Dokument mit klaren Abschnitten, Listen, Tabelle,
Querverweisen und einer reinen Bildseite.
Szenario
Die fiktive Firma Nordstern Logistik betreibt ein automatisiertes Lager. Das Dokument beschreibt
einen Pilotversuch zur Senkung von Fehlkommissionierungen. Es ist so aufgebaut, dass
verschiedene Chunking-Strategien direkt verglichen werden koennen.
Empfohlene Demo-Fragen
1. Was war das Hauptziel des Pilotversu

Metadata on page 0: {'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-07-16T11:45:20+00:00', 'author': 'OpenAI', 'keywords': '', 'moddate': '2026-07-16T11:45:20+00:00', 'subject': '(unspecified)', 'title': 'RAG Chunking Beispieldokument', 'trapped': '/False', 'source': 'chunking.pdf', 'total_pages': 5, 'page': 0, 'page_label': '1'}


# See the Image Problem First-Hand

In [2]:
# Find pages where PyPDF extracted almost no text
empty_pages = [p for p in pages if len(p.page_content.strip()) < 100]
print(f"Pages with little/no text: {len(empty_pages)}")
for p in empty_pages:
    print(f"  Page {p.metadata['page']}: '{p.page_content[:80]}'")

Pages with little/no text: 1
  Page 3: ''


# Naive Fixed-Size Chunking

In [ ]:
from langchain_text_splitters import CharacterTextSplitter

splitter_naive = CharacterTextSplitter(chunk_size=500, chunk_overlap=0,  separator=" " )
naive_chunks = splitter_naive.split_documents(pages)

print(f"Chunks created: {len(naive_chunks)}")
print(f"\nChunk 0:\n{naive_chunks[0].page_content}")
print(f"\nChunk 1:\n{naive_chunks[1].page_content}")

Chunks created: 10

Chunk 0:
RAG Chunking Demo
1
 Beispieldokument fuer RAG-Chunking
Ein kurzes, bewusst heterogenes Dokument mit klaren Abschnitten, Listen, Tabelle,
Querverweisen und einer reinen Bildseite.
Szenario
Die fiktive Firma Nordstern Logistik betreibt ein automatisiertes Lager. Das Dokument beschreibt
einen Pilotversuch zur Senkung von Fehlkommissionierungen. Es ist so aufgebaut, dass
verschiedene Chunking-Strategien direkt verglichen werden koennen.
Empfohlene Demo-Fragen
1. Was war das Hauptziel des

Chunk 1:
des Pilotversuchs?
2. Welche Massnahme hatte den groessten Effekt?
3. Welche Information steht ausschliesslich in der Tabelle?
4. Was zeigt die Bildseite?
5. Welche Aussage wird spaeter eingeschraenkt?
Hinweis: Die Bildseite enthaelt absichtlich keinen eingebetteten Text. Ein rein textbasierter
Parser liefert dort typischerweise einen leeren Chunk.


# Recursive Splitting — The Better Default

In [24]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    separators=["\n\n", "\n", ". ", " ", ""]
)
chunks = splitter.split_documents(pages)

print(f"Chunks created: {len(chunks)}")
print(f"\nChunk 0:\n{chunks[0].page_content}")
print(f"\nChunk metadata: {chunks[0].metadata}")
print(f"\nChunk 1:\n{chunks[1].page_content}")
print(f"\nChunk metadata: {chunks[1].metadata}")

Chunks created: 11

Chunk 0:
RAG Chunking Demo
1
 Beispieldokument fuer RAG-Chunking
Ein kurzes, bewusst heterogenes Dokument mit klaren Abschnitten, Listen, Tabelle,
Querverweisen und einer reinen Bildseite.
Szenario
Die fiktive Firma Nordstern Logistik betreibt ein automatisiertes Lager. Das Dokument beschreibt
einen Pilotversuch zur Senkung von Fehlkommissionierungen. Es ist so aufgebaut, dass
verschiedene Chunking-Strategien direkt verglichen werden koennen.
Empfohlene Demo-Fragen

Chunk metadata: {'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-07-16T11:45:20+00:00', 'author': 'OpenAI', 'keywords': '', 'moddate': '2026-07-16T11:45:20+00:00', 'subject': '(unspecified)', 'title': 'RAG Chunking Beispieldokument', 'trapped': '/False', 'source': 'chunking.pdf', 'total_pages': 5, 'page': 0, 'page_label': '1'}

Chunk 1:
Empfohlene Demo-Fragen
1. Was war das Hauptziel des Pilotversuchs?
2. Welche Massnahme hatte den groessten Effekt?
3

# 6. Add Custom Metadata

In [16]:
for i, chunk in enumerate(chunks):
    chunk.metadata["chunk_index"] = i
    chunk.metadata["document_category"] = "company-policy"
    chunk.metadata["char_count"] = len(chunk.page_content)

print(chunks[5].metadata)

{'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-07-16T11:45:20+00:00', 'author': 'OpenAI', 'keywords': '', 'moddate': '2026-07-16T11:45:20+00:00', 'subject': '(unspecified)', 'title': 'RAG Chunking Beispieldokument', 'trapped': '/False', 'source': 'chunking.pdf', 'total_pages': 5, 'page': 0, 'page_label': '1', 'chunk_index': 5, 'document_category': 'company-policy', 'char_count': 49}


# Local Embeddings with sentence-transformers

In [17]:
from langchain_community.embeddings import HuggingFaceEmbeddings

local_embedder = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

sample_text = chunks[0].page_content
vector = local_embedder.embed_query(sample_text)

print(f"Vector dimensions: {len(vector)}")
print(f"First 10 values: {vector[:10]}")

/tmp/ipykernel_366068/1309799984.py:3: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  local_embedder = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 9318.06it/s]


Vector dimensions: 384
First 10 values: [-0.10966917127370834, 0.045132946223020554, 0.0562039315700531, -0.0764738991856575, -0.028863774612545967, -0.03676290437579155, 0.057140789926052094, -0.0017656012205407023, -0.07106068730354309, -0.02237917296588421]


# 8. OpenAI Embeddings for Comparison

In [18]:
from langchain_openai import OpenAIEmbeddings

openai_embedder = OpenAIEmbeddings(model="text-embedding-3-small")

vector_openai = openai_embedder.embed_query(sample_text)
print(f"OpenAI vector dimensions: {len(vector_openai)}")

OpenAI vector dimensions: 1536


# Store in Chroma with Metadata

In [8]:
import chromadb
from langchain_community.vectorstores import Chroma

# Use local embedder for the demo (no API key needed)
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=local_embedder,
    persist_directory="./chroma_db",
    collection_name="company-docs"
)

print(f"Documents in store: {vectorstore._collection.count()}")

Documents in store: 104


#  Inspect What’s Stored

In [9]:
# Peek at what's actually in the store
collection = vectorstore._collection
results = collection.get(limit=3, include=["documents", "metadatas"])

for i in range(len(results["documents"])):
    print(f"\n--- Chunk {i} ---")
    print(f"Text: {results['documents'][i][:150]}...")
    print(f"Metadata: {results['metadatas'][i]}")


--- Chunk 0 ---
Text: Provided proper attribution is provided, Google hereby grants permission to
reproduce the tables and figures in this paper solely for use in journalis...
Metadata: {'subject': '', 'author': '', 'moddate': '2024-04-10T21:11:43+00:00', 'char_count': 492, 'chunk_index': 0, 'creator': 'LaTeX with hyperref', 'document_category': 'company-policy', 'page': 0, 'trapped': '/False', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'total_pages': 15, 'producer': 'pdfTeX-1.40.25', 'title': '', 'keywords': '', 'creationdate': '2024-04-10T21:11:43+00:00', 'source': 'sample.pdf', 'page_label': '1'}

--- Chunk 1 ---
Text: University of Toronto
aidan@cs.toronto.edu
Łukasz Kaiser∗
Google Brain
lukaszkaiser@google.com
Illia Polosukhin∗ ‡
illia.polosukhin@gmail.com
Abstract...
Metadata: {'subject': '', 'title': '', 'author': '', 'trapped': '/False', 'page_label': '1', 'total_pages': 15, 'page': 0, 'char_count': 454, 'keywor